## Experiment Ariadne

Aim is to correlate OOD-deltas with ID-deltas. For this, we'll collect Hard negative examples from the rollouts per checkpoints - questions which contain a calculation error. To filter eligible traces, we'll look for subsets of traces which have non-zero-solverate, s.t. we obtain ID-gts and ID-negative samples.

Later on, we'll pipe the negative samples through a judge model to decide whether there is a calculation error present which lead to the wrong result. For those samples, we'll create tuples $(gt_{ID}, aug_{ID})$ which we use to measure the in-distribution delta between making a calculation error and correctly solving the task.

#### 09.12 Update
- removing negative samples which contain "```python" keyword, as python-code hallucination seems to be prevalent in Qwen2.5-7B generated answers.
- removing answers which aren't parsable, i.e. which do not contain "\\boxed{}" in last 300 characters

#### 10.12 Update
- making the prompt way simpler -> removing examples, making prompt slim

In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
path = "/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/qwen2.5_7b__gspo/val_jsonl"
checkpoints = []
for file in os.listdir(path):
    with open(os.path.join(path, file), 'r') as jsonfile:
        cp = int(file.split('_')[0])
        df = pd.read_json(jsonfile, lines=True)
        df['checkpoint'] = [cp] * len(df)
        checkpoints.append(df)

In [3]:
def filter_nonzero_std_and_onlymath500(dfs : list[pd.DataFrame]) -> list[pd.DataFrame]:
    """
        Filters a list of dataframes to only include samples which have non-zero std
        rewards -> which are solved and not solved in the same set of answers.
    """
    ret = []
    for df in dfs:
        df = df.iloc[:500 * 256]
        mask = df.groupby('input', sort=False)['score'].transform(lambda x: x.std() != 0)
        candidates = df[mask]
        ret.append(candidates)
    return ret

In [4]:
dfs = filter_nonzero_std_and_onlymath500(checkpoints)

In [5]:
[len(df) for df in dfs]

[51200, 118528, 56832, 83712, 63232, 115968, 69120, 56832, 59648]

In [6]:
import warnings
from functools import wraps
def ignore_warnings(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return func(*args, **kwargs)
    return wrapper

def filter_overlong_sequences(dfs: list[pd.DataFrame], max_resp_len : int = 1024) -> list[pd.DataFrame]:
    ret = []
    for df in dfs:
        approx_tokens = (df['input'].apply(len) + df['output'].apply(len)) / 3
        ret.append(df[approx_tokens < max_resp_len])
    return ret

@ignore_warnings
def filter_negative_samples(dfs : list[pd.DataFrame], keep_ans_per_q : int = 5) -> list[pd.DataFrame]:
    """
        Given a list of pandas DataFrames this function returns the same dataframes, but
        only questions which are negatively answered.

        It retains the initial question index and row index of the response to later match
        back.

        Also: we're filtering out sequences which contain '```python' or do not contain '\\boxed{..}' within the
        last 300 characters.
    """
    ret = []
    for df in dfs:
        negdf = df[df['score'] <= 0]

        cond = negdf['output'].apply(lambda x: "python" not in x and '\\boxed' in x[-100:])
        negdf = negdf[cond]
        sampled = (
            negdf.groupby("input", sort=False, group_keys=False)
            .apply(lambda g: g.sample(n=min(len(g), keep_ans_per_q), random_state=0))
        )
        ret.append(sampled)
    return ret

In [7]:
negatives = filter_negative_samples(filter_overlong_sequences(dfs)) # index is now

In [8]:
list(map(len, negatives))

[825, 1674, 996, 1498, 1096, 1760, 1227, 938, 1048]

In [9]:
negatives[0].head(2)

,input,output,gts,score,step,reward,acc,checkpoint
364,system\nYou are a helpful assistant.\nuser\nDe...,To find a way to write the double sum \(\sum_{...,p - q,0,80,0,0,80
2431,system\nYou are a helpful assistant.\nuser\nTh...,To determine how many different values can be ...,4,0,80,0,0,80


In [10]:
inp = negatives[0].iloc[0]['output']
inp

"To find a way to write the double sum \\(\\sum_{j=1}^\\infty \\sum_{k=1}^\\infty \\frac{1}{(j+k)^3}\\) in terms of \\(p = \\sum_{k=1}^\\infty \\frac{1}{k^2}\\) and \\(q = \\sum_{k=1}^\\infty \\frac{1}{k^3}\\), we will proceed step by step.\n\nFirst, let's denote the double sum by \\(S\\):\n\\[ S = \\sum_{j=1}^\\infty \\sum_{k=1}^\\infty \\frac{1}{(j+k)^3}. \\]\n\nTo simplify this, we can change the order of summation. Consider the sum over all possible values of \\(n = j + k\\). For a fixed \\(n\\), \\(j\\) can range from 1 to \\(n-1\\) (since \\(k = n - j\\) must also be a positive integer). Therefore, we can rewrite the double sum as:\n\\[ S = \\sum_{n=2}^\\infty \\sum_{j=1}^{n-1} \\frac{1}{n^3}. \\]\n\nThe inner sum is simply the sum of \\(\\frac{1}{n^3}\\) taken \\(n-1\\) times:\n\\[ \\sum_{j=1}^{n-1} \\frac{1}{n^3} = (n-1) \\cdot \\frac{1}{n^3} = \\frac{n-1}{n^3} = \\frac{1}{n^2} - \\frac{1}{n^3}. \\]\n\nSubstituting this back into the outer sum, we get:\n\\[ S = \\sum_{n=2}^\\in

In [11]:
remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
inp.removeprefix(remove_prefix).removesuffix(remove_suffix)

"To find a way to write the double sum \\(\\sum_{j=1}^\\infty \\sum_{k=1}^\\infty \\frac{1}{(j+k)^3}\\) in terms of \\(p = \\sum_{k=1}^\\infty \\frac{1}{k^2}\\) and \\(q = \\sum_{k=1}^\\infty \\frac{1}{k^3}\\), we will proceed step by step.\n\nFirst, let's denote the double sum by \\(S\\):\n\\[ S = \\sum_{j=1}^\\infty \\sum_{k=1}^\\infty \\frac{1}{(j+k)^3}. \\]\n\nTo simplify this, we can change the order of summation. Consider the sum over all possible values of \\(n = j + k\\). For a fixed \\(n\\), \\(j\\) can range from 1 to \\(n-1\\) (since \\(k = n - j\\) must also be a positive integer). Therefore, we can rewrite the double sum as:\n\\[ S = \\sum_{n=2}^\\infty \\sum_{j=1}^{n-1} \\frac{1}{n^3}. \\]\n\nThe inner sum is simply the sum of \\(\\frac{1}{n^3}\\) taken \\(n-1\\) times:\n\\[ \\sum_{j=1}^{n-1} \\frac{1}{n^3} = (n-1) \\cdot \\frac{1}{n^3} = \\frac{n-1}{n^3} = \\frac{1}{n^2} - \\frac{1}{n^3}. \\]\n\nSubstituting this back into the outer sum, we get:\n\\[ S = \\sum_{n=2}^\\in

In [12]:
# We have to construct the judge prompt from input, output and gts.

def construct_prompt(row : pd.DataFrame) -> list[dict]:
    system = "You are a helpful judge and an expert in mathematical reasoning."
    
    prefix = (
        "You're given a question and an incorrect students answer. Answer '#### yes' if and only if the wrong answer "
        "is caused by an arithmetic error or an algebraic error, i.e. an error involving simplification, summation, multiplication etc. "
        "Answer '#### no' if the wrong answer is caused by another error type e.g. a reasoning error, logic error or false assumptions.\n"
    )
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    student = row['output']
    gt = row['gts']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\nQuestion:\n\n{question}\n Student's incorrect answer:\n\n{student}\nGround Truth Solution:\n\n{gt}\n Please only answer either '#### yes' or '#### no'.\n"
    }]
    return prompt

In [13]:
pdfs = [pd.DataFrame({
    'prompt' : df.apply(construct_prompt, axis=1),
    'step' : df['step'],
    'old_index' : df.index
}) for df in negatives]

In [14]:
[len(df) for df in pdfs]

[825, 1674, 996, 1498, 1096, 1760, 1227, 938, 1048]

In [15]:
pdfs[0].iloc[0]['prompt']

[{'role': 'system',
  'content': 'You are a helpful judge and an expert in mathematical reasoning.'},
 {'role': 'user',
  'content': "You're given a question and an incorrect students answer. Answer '#### yes' if and only if the wrong answer is caused by an arithmetic error or an algebraic error, i.e. an error involving simplification, summation, multiplication etc. Answer '#### no' if the wrong answer is caused by another error type e.g. a reasoning error, logic error or false assumptions.\n\n\nQuestion:\n\nDefine\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} \\quad q = \\sum_{k = 1}^\\infty \\frac{1}{k^3}.\\]Find a way to write\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3}\\]in terms of $p$ and $q.$\n Student's incorrect answer:\n\nTo find a way to write the double sum \\(\\sum_{j=1}^\\infty \\sum_{k=1}^\\infty \\frac{1}{(j+k)^3}\\) in terms of \\(p = \\sum_{k=1}^\\infty \\frac{1}{k^2}\\) and \\(q = \\sum_{k=1}^\\infty \\frac{1}{k^3}\\), we wi

In [16]:
df = pd.concat(pdfs, axis=0)
df = df.reset_index(drop=True)
df.head(2)

,prompt,step,old_index
0,"[{'role': 'system', 'content': 'You are a help...",80,364
1,"[{'role': 'system', 'content': 'You are a help...",80,2431


In [17]:
os.makedirs('/u/rfechner/data/ariadne', exist_ok=True)
with open('/u/rfechner/data/ariadne/id-prompts-simple.parquet', 'wb') as file:
    df.to_parquet(file)